In [1]:
import sys
sys.path.append('../')

import importlib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import linkage

from src.preprocessing import (
    load_cluster_data, clean_customer_data,
    engineer_customer_features, encode_categorical, scale_features
)
import src.utils as _utils
importlib.reload(_utils)
from src.utils import show_figure
import src.display as _display
importlib.reload(_display)
from src.display import (
    init_notebook_theme, show_hero, show_section, show_insight,
    show_info, show_warning, show_success, show_metrics_row,
    show_findings_list, show_badge, show_table_html, show_architecture_card,
)
init_notebook_theme()

DATA_PATH = '../data/raw/data_cluster.csv'
RANDOM_STATE = 42


In [2]:
show_hero(
    title="Exercice 2 — Segmentation intelligente des clients",
    objective="Identifier des profils clients pour optimiser les campagnes marketing.",
    plan_items=[
        "EDA", "Nettoyage & features", "Prétraitement clustering",
        "Clustering multi-algos", "Évaluation", "Interprétation métier", "Recommandations",
    ],
)
show_section("1. Analyse exploratoire (EDA)")

In [3]:
df = load_cluster_data(DATA_PATH)
show_metrics_row({"Lignes": f"{df.shape[0]:,}", "Colonnes": len(df.columns)})
print(df.columns.tolist())

['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']


In [4]:
print("--- Informations ---")
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n--- Statistiques ---")
df.describe()

--- Informations ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,...,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.0,2240.0,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,49.109375,303.935714,26.302232,166.950000,37.525446,...,5.316518,0.072768,0.074554,0.072768,0.064286,0.013393,0.009375,3.0,11.0,0.149107
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,28.962453,336.597393,39.773434,225.715373,54.628979,...,2.426645,0.259813,0.262728,0.259813,0.245316,0.114976,0.096391,0.0,0.0,0.356274
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,24.000000,23.750000,1.000000,16.000000,3.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,49.000000,173.500000,8.000000,67.000000,12.000000,...,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,74.000000,504.250000,33.000000,232.000000,50.000000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,...,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,11.0,1.000000


In [5]:
from src.charts.segmentation import build_distributions
from src.utils import show_figure

fig = build_distributions(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_distributions.png', width=1100, height=650)


In [6]:
from src.charts.segmentation import build_interactive_exploration
from src.utils import show_figure

fig = build_interactive_exploration(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_interactive_explorer.png', width=1000, height=520)
show_insight(
    "Revenu × dépenses (r ≈ 0,79) : futur Premium ~71 k€ / ~1 233 € vs Digital ~39 k€ / ~178 €. "
    "Âge (~54 vs ~57 ans) et récence (~49 j) ne séparent pas les groupes — seuls revenu et panier structurent le ciblage."
)

In [7]:
from src.charts.segmentation import build_categorical
from src.utils import show_figure

fig = build_categorical(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_categorical.png', width=1000, height=450)
show_insight(
    "Graduation et Married dominent, mais l'éducation seule n'explique pas l'écart de panier ×7 — "
    "TotalSpend et vins discriminent bien mieux que la sociodémographie."
)


In [8]:
from src.charts.segmentation import build_spending_channels
from src.utils import show_figure

fig = build_spending_channels(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_spending_channels.png', width=1000, height=450)
show_insight(
    "Vins ~305 € vs fruits ~26 € (≈ 12×) structurent le portefeuille. "
    "Premium concentrera vins (~610 €) et viandes (~357 €) ; Digital reste sous 100 € sur ces postes."
)


In [9]:
show_section("2. Nettoyage & Feature Engineering")

In [10]:
df_clean = clean_customer_data(df)
df_clean = engineer_customer_features(df_clean)

print(f"Lignes avant nettoyage : {len(df)}")
print(f"Lignes après nettoyage : {len(df_clean)}")
print(f"\nNouvelles features créées : Age, Children, TotalSpend, TotalPurchases")

Lignes avant nettoyage : 2240
Lignes après nettoyage : 2215

Nouvelles features créées : Age, Children, TotalSpend, TotalPurchases


In [11]:
from src.charts.segmentation import build_correlation
from src.utils import show_figure

fig = build_correlation(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_correlation.png', width=1000, height=800)
show_insight(
    "MntWines, MntMeat et TotalSpend corrélés > 0,7 — StandardScaler obligatoire avant K-Means, "
    "sinon Income (~70 k€) écraserait NumWebPurchases (~4 achats)."
)


In [12]:
show_section("3. Prétraitement pour clustering")

In [13]:
# Sélection des features pour le clustering
cat_cols = ['Education', 'Marital_Status']
drop_cols = ['ID', 'Dt_Customer', 'Year_Birth'] + cat_cols

df_encoded = encode_categorical(df_clean, cat_cols)
feature_cols = [c for c in df_encoded.select_dtypes(include=np.number).columns
                if c not in drop_cols]

X = df_encoded[feature_cols].copy()
print(f"Features pour clustering ({len(feature_cols)}) :\n{feature_cols}")

Features pour clustering (28) :
['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response', 'Age', 'Children', 'TotalSpend', 'TotalPurchases']


In [14]:
X_scaled, scaler = scale_features(X)

# PCA pour visualisation 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"Variance expliquée par les 2 premières composantes : {pca.explained_variance_ratio_.sum():.1%}")

# PCA pour réduction dimensionnelle (95% de variance)
pca_full = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)
print(f"Composantes pour 95% variance : {X_pca_full.shape[1]}")

Variance expliquée par les 2 premières composantes : 42.9%
Composantes pour 95% variance : 19


In [15]:
from src.charts.segmentation import build_pca_scree
from src.utils import show_figure

fig = build_pca_scree(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_pca_scree.png', width=1000, height=450)


In [16]:
show_section("4. Clustering")

In [19]:
# Sélection du k optimal (méthode Silhouette sur X_pca_full)
K_RANGE = list(range(2, 11))
silhouette_scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca_full)
    silhouette_scores.append(silhouette_score(X_pca_full, labels))
SILHOUETTE_PEAK_K = K_RANGE[int(np.argmax(silhouette_scores))]
from src.constants import TARGET_CLUSTER_K
BEST_K = TARGET_CLUSTER_K  # k=4 retenu pour 4 profils métier (pic Silhouette à k=2)
print(f"k optimal (Silhouette) : {BEST_K} (score = {max(silhouette_scores):.4f})")

from src.charts.segmentation import build_kmeans_selection
from src.utils import show_figure

fig = build_kmeans_selection(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_kmeans_selection.png', width=1200, height=450)
show_insight(
    f"Silhouette max = {max(silhouette_scores):.3f} à k={SILHOUETTE_PEAK_K} ; production k={BEST_K} (vs {silhouette_scores[1]:.3f} à k=3). "
    "Deux segments capturent l'essentiel sans fragmenter la base en micro-groupes inexploitables."
)


k optimal (Silhouette) : 4 (score = 0.2902)


In [18]:
    from src.constants import TARGET_CLUSTER_K
    SILHOUETTE_PEAK_K = _k_range[int(np.argmax(_scores))]
    BEST_K = TARGET_CLUSTER_K
    print(f"Pic Silhouette : k={SILHOUETTE_PEAK_K} ; k production={BEST_K} (score pic = {max(_scores):.4f})")


NameError: name '_k_range' is not defined

In [ ]:
from src.charts.segmentation import build_clustering_comparison
from src.utils import show_figure

fig = build_clustering_comparison(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_clustering_comparison.png', width=1000, height=800)
show_insight(
    f"DBSCAN : {n_dbscan_clusters} cluster(s), {(dbscan_labels == -1).sum()} points en bruit — inadapté. "
    "K-Means et GMM produisent deux nuages lisibles ; K-Means retenu pour des centroïdes interprétables."
)


In [ ]:
from src.charts.segmentation import build_dendrogram
from src.utils import show_figure

fig = build_dendrogram(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_dendrogram.png', width=1200, height=500)


In [ ]:
show_section("5. Évaluation & comparaison")

In [ ]:
# Tableau de comparaison
eval_results = []
valid_algos = [
    ('K-Means', kmeans_labels),
    ('Agglomerative', agglo_labels),
    ('GMM', gmm_labels),
]
if n_dbscan_clusters > 1:
    mask = dbscan_labels != -1
    valid_algos.append(('DBSCAN', dbscan_labels))

for name, labels in valid_algos:
    try:
        sil = silhouette_score(X_pca_full, labels)
        db = davies_bouldin_score(X_pca_full, labels)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        eval_results.append({'Algorithme': name, 'N clusters': n_clust,
                              'Silhouette ↑': round(sil, 4), 'Davies-Bouldin ↓': round(db, 4)})
    except Exception as e:
        print(f"{name} : {e}")

eval_df = pd.DataFrame(eval_results).set_index('Algorithme')
print(eval_df.to_string())

In [ ]:
show_section("6. Interprétation métier des profils")

In [ ]:
# Ajouter les labels K-Means au dataframe original
df_result = df_clean.copy()
df_result['Cluster'] = kmeans_labels

# Profil moyen par cluster
profile_cols = ['Income', 'Age', 'TotalSpend', 'TotalPurchases',
                'MntWines', 'MntMeatProducts', 'NumWebPurchases',
                'NumStorePurchases', 'Recency', 'Children']
profile_cols = [c for c in profile_cols if c in df_result.columns]

cluster_profiles = df_result.groupby('Cluster')[profile_cols].mean().round(1)
print("=== Profils moyens par cluster ===")
print(cluster_profiles.T.to_string())

c0, c1 = cluster_profiles.loc[0], cluster_profiles.loc[1]
spend_ratio = c1['TotalSpend'] / c0['TotalSpend']
show_insight(
    f"Cluster 0 : revenu {c0['Income']:,.0f} €, panier {c0['TotalSpend']:.0f} €, web {c0['NumWebPurchases']:.1f}. "
    f"Cluster 1 : revenu {c1['Income']:,.0f} €, panier {c1['TotalSpend']:.0f} € (×{spend_ratio:.1f}), "
    f"web {c1['NumWebPurchases']:.1f} — Premium achète plus sur le web, pas Digital."
)


In [ ]:
from src.charts.segmentation import build_cluster_profiles
from src.utils import show_figure

fig = build_cluster_profiles(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_cluster_profiles.png', width=1100, height=500)
show_insight(
    f"Heatmap : écart revenu ×{c1['Income']/c0['Income']:.1f}, panier ×{spend_ratio:.0f}, "
    f"vins ×{c1['MntWines']/c0['MntWines']:.0f}. Digital se distingue surtout par plus d'enfants ({c0['Children']:.1f} vs {c1['Children']:.1f})."
)


In [ ]:
# Nommage automatique aligné sur src/training.py et l'API
from src.training import _assign_cluster_labels
from src.constants import CLUSTER_API_COLUMNS

centroids_api = cluster_profiles[[c for c in CLUSTER_API_COLUMNS if c in cluster_profiles.columns]]
cluster_labels = _assign_cluster_labels(centroids_api)
df_result['Profil'] = df_result['Cluster'].map(cluster_labels)

print("Labels attribués :", cluster_labels)
print("Taille des clusters :")
print(df_result['Cluster'].value_counts().sort_index())

sizes = df_result['Cluster'].value_counts().sort_index()
show_insight(
    f"{cluster_labels[1]} : {sizes[1]} clients ({100*sizes[1]/len(df_result):.0f} %), panier {c1['TotalSpend']:.0f} € — fidélité haut de gamme. "
    f"{cluster_labels[0]} : {sizes[0]} clients ({100*sizes[0]/len(df_result):.0f} %), panier {c0['TotalSpend']:.0f} € — segment masse ; "
    "le libellé « Digital » vient du nommage auto, pas d'une dominance web."
)

In [ ]:
from src.charts.segmentation import build_radar_profiles
from src.utils import show_figure

fig = build_radar_profiles(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_radar_profiles.png', width=1000, height=500)
show_insight(
    f"Premium « plein » sur revenu/dépenses ; Digital faible partout sauf Children ({c0['Children']:.1f}). "
    f"Récence identique (~{c0['Recency']:.0f} j) — inutilisable seule pour différencier les campagnes."
)


In [ ]:
show_section("7. Recommandations business")

In [ ]:
from src.charts.segmentation import build_campaign_response
from src.utils import show_figure

fig = build_campaign_response(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_campaign_response.png', width=900, height=450)

cmp_cols = [c for c in df_result.columns if c.startswith('AcceptedCmp') or c == 'Response']
if cmp_cols:
    df_result['CampaignResponse'] = df_result[cmp_cols].sum(axis=1)
    by_profil = df_result.groupby('Profil')['CampaignResponse'].mean()
    show_insight(
        "Taux d'acceptation campagne : "
        + ", ".join(f"{p} {100*v:.1f} %" for p, v in by_profil.items())
        + ". Prioriser les envois sur le segment le plus réceptif plutôt que les 85 % de refus global."
    )


In [ ]:
show_section("Synthèse Exercice 2")
_actions = {
    "Premium": "Fidélité, offres exclusives",
    "Digital": "Email, app mobile, promotions web",
    "Promo-sensible": "Coupons ciblés",
    "Dormant": "Campagne de réactivation",
}
profiles_df = pd.DataFrame([
    {"Cluster": k, "Profil": v, "Actions": _actions.get(v, "—")}
    for k, v in sorted(cluster_labels.items())
    if k < BEST_K
])
show_table_html(profiles_df.set_index("Cluster"), title="Profils identifiés")
show_findings_list([
    f"k={BEST_K} retenu pour 4 profils (pic Silhouette {max(silhouette_scores):.3f} à k={SILHOUETTE_PEAK_K if 'SILHOUETTE_PEAK_K' in dir() else 2}) — Premium {sizes[1]} clients, panier {c1['TotalSpend']:.0f} €",
    f"Digital {sizes[0]} clients ({100*sizes[0]/len(df_result):.0f} %), panier {c0['TotalSpend']:.0f} € — segment masse, pas « plus web »",
    "Campagnes différenciées : fidélité Premium vs promotions modérées Digital ; recalcul mensuel + suivi Silhouette",
], title="Recommandations stratégiques")